# VigiLense - weapon detector v2 (all types of guns + knives)
Merges several public weapon datasets, maps every class name to one list, trains YOLO26n, saves to Google Drive.
Unified classes: **pistol, rifle, knife, blade** (machete / sword / axe). Rare classes are dropped automatically.
Run cells top to bottom. Runtime -> T4 GPU. Sources: Dangerous Items (Zenodo, no login), Roboflow Pistol-Rifle-Knife (free API key), optional Kaggle datasets (kaggle.json).

In [ ]:
# 1) GPU, packages, Drive
import subprocess, sys
print(subprocess.run(["nvidia-smi","-L"],capture_output=True,text=True).stdout or "NO GPU - change runtime to T4")
!pip -q install ultralytics roboflow kaggle
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 2) Settings
QUICK_TEST = True          # True = 3 epochs on 10% of data (pipeline check). Set False for the real run.
EPOCHS     = 3 if QUICK_TEST else 80
RUN_NAME   = "weapons_v2_quick" if QUICK_TEST else "weapons_v2"
MODEL      = "yolo26n.pt"
MIN_BOXES  = 300           # a class with fewer boxes in the training split is dropped

USE_DANGEROUS_ITEMS = True   # Zenodo, no login
USE_ROBOFLOW        = True   # needs a free API key (roboflow.com -> Settings -> API Keys)
KAGGLE_DATASETS     = []     # e.g. ["snehilsanyal/weapon-detection-test"]  (needs kaggle.json)

UNIFIED = ["pistol","rifle","knife","blade"]
# keyword -> unified class (first match wins). Edit if cell 4 prints names that are dropped by mistake.
KEYWORDS = [
 ("pistol","pistol"),("handgun","pistol"),("revolver","pistol"),("gun","pistol"),("firearm","pistol"),
 ("rifle","rifle"),("shotgun","rifle"),("ak47","rifle"),("ak-47","rifle"),("assault","rifle"),("smg","rifle"),("sniper","rifle"),("machine","rifle"),
 ("knife","knife"),("dagger","knife"),("blade","knife"),("cutter","knife"),
 ("machete","blade"),("sword","blade"),("axe","blade"),("hatchet","blade"),("cleaver","blade"),
]
# for datasets whose yaml has no names: {source: [name for id 0, id 1, ...]}
FALLBACK_NAMES = {"dangerous": ["machete","knife","baseball bat","rifle","gun"]}

In [ ]:
# 3) Download the sources
import os, shutil, hashlib, zipfile
from pathlib import Path
SRC = Path("/content/src"); SRC.mkdir(exist_ok=True)
CACHE = Path("/content/drive/MyDrive/vigilense/datasets"); CACHE.mkdir(parents=True, exist_ok=True)

if USE_DANGEROUS_ITEMS:
    z = CACHE/"Dangerous Items.zip"
    if not z.exists():
        subprocess.run(["curl","-L","--fail","-o",str(z),"https://zenodo.org/records/16422779/files/Dangerous%20Items.zip?download=1"],check=True)
    assert hashlib.md5(z.read_bytes()).hexdigest()=="f567c3b6e070b2c36578a09c5c7eb9e2","Dangerous Items download incomplete: delete the zip in Drive and rerun"
    if not (SRC/"dangerous").exists():
        zipfile.ZipFile(z).extractall(SRC/"dangerous")
    print("dangerous OK")

if USE_ROBOFLOW:
    import getpass
    from roboflow import Roboflow
    key = getpass.getpass("Roboflow API key (hidden): ")
    rf = Roboflow(api_key=key)
    rf.workspace("pistolrifle").project("pistol-rifle-knife").version(1).download("yolov8", location=str(SRC/"roboflow_prk"))
    print("roboflow OK")

if KAGGLE_DATASETS:
    from google.colab import files
    if not Path("/root/.kaggle/kaggle.json").exists():
        up = files.upload()                                   # choose kaggle.json
        os.makedirs("/root/.kaggle", exist_ok=True); shutil.move("kaggle.json","/root/.kaggle/kaggle.json"); os.chmod("/root/.kaggle/kaggle.json",0o600)
    for ds in KAGGLE_DATASETS:
        out = SRC/("kaggle_"+ds.split("/")[-1])
        subprocess.run(["kaggle","datasets","download","-d",ds,"-p",str(out),"--unzip"],check=True)
        print(ds,"OK")
print(sorted(p.name for p in SRC.iterdir()))

In [ ]:
# 4) Inspect: find images/labels/class names in every source
import yaml, re, random
IMG_EXT = {".jpg",".jpeg",".png",".bmp",".webp"}

def find_images(root):
    return [p for p in Path(root).rglob("*") if p.suffix.lower() in IMG_EXT]

def label_for(img, root):
    parts = list(img.parts)
    for i in range(len(parts)-1, -1, -1):          # swap the last "images" folder for "labels"
        if parts[i].lower()=="images":
            lp = Path(*parts[:i], "labels", *parts[i+1:]).with_suffix(".txt"); return lp if lp.exists() else None
    lp = img.with_suffix(".txt"); return lp if lp.exists() else None

def split_of(img):
    s = [p.lower() for p in img.parts]
    for key,name in (("train","train"),("valid","val"),("val","val"),("test","test")):
        if key in s: return name
    return None

def source_names(root, key):
    for y in Path(root).rglob("*.y*ml"):
        d = yaml.safe_load(open(y, encoding="utf-8", errors="ignore"))
        n = d.get("names") if isinstance(d,dict) else None
        if n: return [n[i] for i in sorted(n)] if isinstance(n,dict) else list(n)
    for k,v in FALLBACK_NAMES.items():
        if k in key: return v
    return None

def to_unified(name):
    n = name.lower()
    for kw,u in KEYWORDS:
        if kw in n: return UNIFIED.index(u)
    return None

items, report = [], {}
for src in sorted(SRC.iterdir()):
    names = source_names(src, src.name)
    imgs = [(i, label_for(i, src)) for i in find_images(src)]
    imgs = [(i,l) for i,l in imgs if l is not None]
    print(f"\n== {src.name}: {len(imgs)} labelled images | class names: {names}")
    if names is None:
        print("   !! no class names found - add them to FALLBACK_NAMES and rerun"); continue
    cmap = {i: to_unified(n) for i,n in enumerate(names)}
    print("   mapping:", {names[i]: (UNIFIED[u] if u is not None else "DROPPED") for i,u in cmap.items()})
    for i,l in imgs: items.append((src.name, i, l, cmap, split_of(i)))
print("\ntotal labelled images:", len(items))

In [ ]:
# 5) Build the merged dataset (keeps each source's own train/val/test split; random 80/10/10 only where a source has none)
OUT = Path("/content/weapons_v2"); shutil.rmtree(OUT, ignore_errors=True)
random.seed(0)
rows = []
for s,i,l,cmap,sp in items:
    boxes = []
    for line in open(l):
        p = line.split()
        if len(p)>=5 and int(float(p[0])) in cmap and cmap[int(float(p[0]))] is not None:
            boxes.append((cmap[int(float(p[0]))], *p[1:5]))
    rows.append((s,i,boxes,sp))
nosplit = [r for r in rows if r[3] is None]
random.shuffle(nosplit)
for k,r in enumerate(nosplit):
    r_sp = "train" if k < 0.8*len(nosplit) else ("val" if k < 0.9*len(nosplit) else "test")
    rows[rows.index(r)] = (r[0],r[1],r[2],r_sp)

from collections import Counter
cnt = Counter()
for s,i,b,sp in rows:
    if sp=="train":
        cnt.update(c for c,*_ in b)
keep = [c for c in range(len(UNIFIED)) if cnt[c] >= MIN_BOXES]
print("train boxes per class:", {UNIFIED[c]:cnt[c] for c in range(len(UNIFIED))})
print("kept classes:", [UNIFIED[c] for c in keep], "| dropped:", [UNIFIED[c] for c in range(len(UNIFIED)) if c not in keep])
newid = {c:k for k,c in enumerate(keep)}
names = [UNIFIED[c] for c in keep]

n_img = Counter()
for s,i,b,sp in rows:
    b2 = [(newid[c],*xy) for c,*xy in b if c in newid]
    if not b2 and random.random() > 0.1:            # keep only ~10% of background images
        continue
    if QUICK_TEST and random.random() > 0.1: continue
    for d in ("images","labels"): (OUT/sp/d).mkdir(parents=True, exist_ok=True)
    stem = f"{s}_{i.stem}"
    shutil.copy(i, OUT/sp/"images"/(stem+i.suffix.lower()))
    open(OUT/sp/"labels"/(stem+".txt"),"w").write("\n".join(" ".join(map(str,x)) for x in b2))
    n_img[sp]+=1
yaml.safe_dump({"path":str(OUT),"train":"train/images","val":"val/images","test":"test/images","names":dict(enumerate(names))}, open(OUT/"data.yaml","w"))
print("images:", dict(n_img), "| names:", names)
assert n_img["train"]>0 and n_img["val"]>0 and n_img["test"]>0, "a split is empty"

In [ ]:
# 6) Train
from ultralytics import YOLO
model = YOLO(MODEL)
model.train(data=str(OUT/"data.yaml"), epochs=EPOCHS, imgsz=640, batch=16, patience=20,
            project="/content/drive/MyDrive/vigilense/runs", name=RUN_NAME, exist_ok=True, workers=2, plots=True)

In [ ]:
# 7) Test-set results (report these)
best = f"/content/drive/MyDrive/vigilense/runs/{RUN_NAME}/weights/best.pt"
m = YOLO(best).val(data=str(OUT/"data.yaml"), split="test", imgsz=640, plots=True)
print(f"mAP50: {m.box.map50:.3f} | mAP50-95: {m.box.map:.3f}\nPrecision: {m.box.mp:.3f} | Recall: {m.box.mr:.3f}")
for k,n in enumerate(m.names.values()):
    print(f"  {n:8s} P {m.box.p[k]:.3f}  R {m.box.r[k]:.3f}  mAP50 {m.box.ap50[k]:.3f}")

In [ ]:
# 8) Sample predictions
import glob
from IPython.display import Image, display
res = YOLO(best).predict(random.sample(glob.glob(str(OUT/"test/images/*")), 6), conf=0.4, save=True, project="/content/runs", name="v2_samples", exist_ok=True, verbose=False)
for f in sorted(glob.glob("/content/runs/v2_samples/*"))[:6]: display(Image(f, width=420))

**If Colab disconnects:** reconnect, run cells 1-5 again (data is cached on Drive / re-downloaded), then run this and skip cell 6:
`YOLO('/content/drive/MyDrive/vigilense/runs/weapons_v2/weights/last.pt').train(resume=True)`

**After training:** download `best.pt` from Drive (`vigilense/runs/weapons_v2/weights/`) to `C:\vigilense\weights\weapons_v2.pt`, then run the app with `--weapon-model weights/weapons_v2.pt`.